# ML-10 — Action Playbook & Recommendation Engine

This notebook translates the model probabilities into actionable human workflows, exporting the final queue for the research paper.

## 1. Ranked actions + reason codes

The model outputs a `decay_probability`. We will translate this into clear reason codes based on feature thresholds so the editorial team knows *why* a page was flagged.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data & Train Model (Reproducing Week 5 state)
data_path = '../../../flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = '../../data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(data_path)
df = df[df['impressions_90d'] > 100].copy() # Visibility filter

features = ['content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

X = df[features].fillna(0)
y = df['is_declining']

model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
model.fit(X, y)
df['decay_probability'] = model.predict_proba(X)[:, 1]

# 2. Assign Reason Codes
df['reason_code'] = 'generic_decay'
df.loc[(df['decay_probability'] > 0.6) & (df['days_since_last_update'] > 180), 'reason_code'] = 'stale_content_decay'
df.loc[(df['decay_probability'] > 0.6) & (df['avg_position'] > 10), 'reason_code'] = 'page_2_decay'

print("Playbook categories generated.")

Playbook categories generated.


## 2. Intended use and limits

**Intended Use:** This tool generates a weekly prioritized queue to help content managers decide which pages to rewrite, ensuring editorial budget is spent defending high-traffic assets.
**Limits:** The model cannot read the text of the page. It does not know if a page is factually outdated or just suffering a seasonal drop. It is strictly a directional triage tool.

In [2]:
print("Limits defined: Directional triage tool, not an automated publishing engine.")

Limits defined: Directional triage tool, not an automated publishing engine.


## 3. Human review + the no-go list

**Human Review Required:** A human editor must open the top 50 URLs and determine *how* to rewrite them (e.g., updating stats, adding a video, merging with another post).
**The NO-GO List (What NOT to automate):** We will never automate the actual rewriting and publishing of content directly to the CMS based on this score, as AI hallucinations could ruin evergreen assets.

In [3]:
print("No-Go List established: Automated publishing is strictly prohibited.")

No-Go List established: Automated publishing is strictly prohibited.


## 4. Monitoring / retrain triggers

**Monitoring:** We will monitor the `Precision@50` metric monthly using an updated data snapshot.
**Retrain Triggers:** If Precision@50 drops below 60% for two consecutive months, or if Google announces a major core update that shifts ranking paradigms, the model will be retrained on fresh data.

In [4]:
print("Monitoring plan documented.")

Monitoring plan documented.


## 5. Exports for the paper

I am now exporting the ranked queue to `work/outputs/action_queue.csv` and saving a feature importance chart for the capstone paper.

In [5]:
import matplotlib.pyplot as plt

# Export the queue
queue = df[df['decay_probability'] > 0.5].sort_values('decay_probability', ascending=False)
queue_export = queue[['content_id', 'decay_probability', 'reason_code', 'impressions_90d', 'days_since_last_update']]

os.makedirs('../outputs', exist_ok=True)
queue_export.to_csv('../outputs/action_queue.csv', index=False)

# Generate & Export a simple Feature Importance Chart
os.makedirs('../figures', exist_ok=True)
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#2563EB')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../figures/feature_importance.png')
plt.close()

print("Exported queue to work/outputs/action_queue.csv")
print("Exported chart to work/figures/feature_importance.png")

Exported queue to work/outputs/action_queue.csv
Exported chart to work/figures/feature_importance.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.